# Práctica 4. Seguimiento, conteo, OCR y CSV de entrega

Un único cuaderno que, con **Ejecutar todo** (kernel del entorno `VC_P4`, abierto desde la carpeta `P4`), produce en `resultado_seguimiento/`:

1. `seguimiento.mp4`: vídeo anotado con cajas, IDs de seguimiento, matrículas, su texto leído y el contador por clase,
2. `detecciones.csv`: una fila por objeto y fotograma, con el formato que pide el enunciado,
3. `conteo.csv`: objetos **únicos** por clase (IDs distintos, no detecciones),
4. `matriculas_leidas.csv` y `matriculas/`: el texto más votado y el mejor recorte de la matrícula de cada vehículo,
5. extras: `flujo.csv` (dirección de cada objeto) y `seguimiento_anonimo.mp4` (personas y matrículas difuminadas).

Flujo: pasada 1 (`track()` de personas y vehículos con ByteTrack + detector de matrículas) → **unión de pistas** para corregir los cambios de ID → **pasada OCR** (EasyOCR en cada fotograma + SmolVLM sobre los mejores recortes de cada vehículo) → pasada 2 (vídeo, CSV y recortes con los IDs ya unificados) → conteo y flujo.

Tarda unos 15 minutos con la GPU del equipo de pruebas (RTX 3050).

In [1]:
import os, sys
from pathlib import Path
# Sin "conda activate", faltan las DLL de SSL y EasyOCR no puede descargar sus modelos
os.environ["PATH"] = os.pathsep.join([sys.prefix, str(Path(sys.prefix) / "Library" / "bin"), os.environ["PATH"]])

import csv
import heapq
import re
import time
from collections import Counter, defaultdict

import cv2
from ultralytics import YOLO

## Configuración

In [2]:
VIDEO = Path("../datos/videos/C0142.MP4")
PESOS_MATRICULAS = Path("runs/matriculas_yolo11n_lejanas/weights/best.pt")
PESOS_OBJETOS = "yolo11n.pt"

SALIDA = Path("resultado_seguimiento")
VIDEO_OUT = SALIDA / "seguimiento.mp4"
VIDEO_ANON = SALIDA / "seguimiento_anonimo.mp4"
CSV_OUT = SALIDA / "detecciones.csv"
CONTEO_OUT = SALIDA / "conteo.csv"
MATRICULAS_OUT = SALIDA / "matriculas_leidas.csv"
FLUJO_OUT = SALIDA / "flujo.csv"
RECORTES = SALIDA / "matriculas"
RECORTES.mkdir(parents=True, exist_ok=True)

# Detección (clases COCO)
CLASES = {0: "person", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
VEHICULOS = {"car", "motorcycle", "bus", "truck"}
# Clases que el detector confunde entre sí: se tratan como el mismo objeto al unir pistas
GRUPO = {"person": "persona", "motorcycle": "moto", "car": "vehiculo", "truck": "vehiculo", "bus": "vehiculo"}
IMGSZ_OBJETOS = 1280
CONF_OBJETOS = 0.10      # bajo a propósito: ByteTrack usa las detecciones débiles para recuperar oclusiones
IMGSZ_MATRICULAS = 1920  # resolución completa, para no encoger las matrículas lejanas
CONF_MATRICULAS = 0.25

# Unión de pistas
MAX_HUECO = 300   # fotogramas que un objeto puede estar tapado y seguir siendo el mismo
IOU_MIN = 0.30
MIN_FRAMES = 10   # los objetos que aparecen menos fotogramas no cuentan

# OCR
MARGEN_RECORTE = 0.10
USAR_OCR = True
OCR_MIN_ANCHO = 20
OCR_ANCHO_OBJETIVO = 240
USAR_VLM = True
VLM_MODELO = "HuggingFaceTB/SmolVLM-500M-Instruct"
VLM_CANDIDATOS = 7
VLM_MIN_ANCHO = 40
VLM_MIN_VOTOS = 2  # una lectura suelta del VLM no basta (salvo que coincida con EasyOCR)

ANONIMIZAR = True
SHOW = False
MAX_FRAMES = int(os.environ.get("MAX_FRAMES", 0)) or None  # para pruebas rápidas

assert VIDEO.exists(), f"No existe el vídeo: {VIDEO.resolve()}"
assert PESOS_MATRICULAS.exists(), f"No existen los pesos: {PESOS_MATRICULAS.resolve()}"

# ByteTrack con más memoria (track_buffer) y umbrales más bajos que los de por defecto
TRACKER = SALIDA / "bytetrack_custom.yaml"
TRACKER.write_text(
    "tracker_type: bytetrack\n"
    "track_high_thresh: 0.30\n"
    "track_low_thresh: 0.10\n"
    "new_track_thresh: 0.40\n"
    "track_buffer: 150\n"
    "match_thresh: 0.85\n"
    "fuse_score: True\n"
)

148

In [3]:
modelo_objetos = YOLO(PESOS_OBJETOS)
modelo_matriculas = YOLO(str(PESOS_MATRICULAS))
print("Matrículas:", modelo_matriculas.names)

Matrículas: {0: 'License Plate'}


## Asociación matrícula-vehículo

Una matrícula detectada se asigna al vehículo que la contiene. Se exigen las mismas restricciones geométricas que en
`VC_P4_Yolo.ipynb` (relación de aspecto de matrícula, zona baja del vehículo, tamaño razonable). Si varias
matrículas caen en el mismo vehículo, se conserva la de mayor confianza. Las matrículas que no encajan en ningún
vehículo se descartan como falsos positivos.

In [4]:
def cabe_en_vehiculo(caja, veh):
    """Comprueba que la caja tiene forma de matrícula y está en la parte baja del vehículo `veh`."""
    x1, y1, x2, y2 = caja
    ancho, alto = x2 - x1, y2 - y1
    if alto <= 0 or not 1.2 <= ancho / alto <= 6.5:
        return False
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    vx1, vy1, vx2, vy2 = veh
    v_ancho, v_alto = vx2 - vx1, vy2 - vy1
    return (vx1 <= cx <= vx2
            and vy1 + 0.35 * v_alto <= cy <= vy2 + 0.05 * v_alto
            and ancho <= 0.65 * v_ancho)


def asociar_matriculas(placas, vehiculos):
    """
    placas:    lista de (caja, conf)
    vehiculos: lista de (track_id, caja)
    Devuelve {track_id: (caja_matricula, conf)} con la mejor matrícula de cada vehículo.
    Si una matrícula cabe en varios vehículos (solapados), se asigna al de menor área.
    """
    asignadas = {}
    for caja, conf in placas:
        candidatos = [(tid, v) for tid, v in vehiculos if cabe_en_vehiculo(caja, v)]
        if not candidatos:
            continue
        tid, _ = min(candidatos, key=lambda tv: (tv[1][2] - tv[1][0]) * (tv[1][3] - tv[1][1]))
        if tid not in asignadas or conf > asignadas[tid][1]:
            asignadas[tid] = (caja, conf)
    return asignadas

## Pasada 1: detección y seguimiento

Se recorre el vídeo y se guardan en memoria las detecciones con el ID bruto del tracker, sin dibujar nada todavía.
`persist=True` mantiene los IDs entre llamadas a `track`.

**Por qué dos pasadas.** Cuando un coche cruza por delante de otros aparcados, el tracker pierde a los aparcados y,
al reaparecer, les da un ID nuevo (*ID switch*). Para corregirlo hace falta ver el vídeo entero antes de decidir
qué IDs son el mismo objeto, y solo entonces dibujar y escribir el CSV con los IDs ya unificados.

In [5]:
objs_por_frame = {}    # fotograma -> [(id_bruto, clase, conf, caja)]
placas_por_frame = {}  # fotograma -> [(caja, conf)]

vid = cv2.VideoCapture(str(VIDEO))
total = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))
n = 0
while True:
    ret, img = vid.read()
    if not ret or (MAX_FRAMES and n >= MAX_FRAMES):
        break
    n += 1

    r = modelo_objetos.track(img, persist=True, imgsz=IMGSZ_OBJETOS, conf=CONF_OBJETOS,
                             classes=list(CLASES), tracker=str(TRACKER), verbose=False)[0]
    objs = []
    if r.boxes.id is not None:   # es None mientras el tracker no ha confirmado ninguna pista
        for tid, cls, conf, caja in zip(r.boxes.id.int().tolist(), r.boxes.cls.int().tolist(),
                                        r.boxes.conf.tolist(), r.boxes.xyxy.tolist()):
            objs.append((tid, CLASES[cls], conf, caja))
    objs_por_frame[n] = objs

    p = modelo_matriculas(img, imgsz=IMGSZ_MATRICULAS, conf=CONF_MATRICULAS, verbose=False)[0].boxes
    placas_por_frame[n] = list(zip(p.xyxy.tolist(), p.conf.tolist()))

    if n % 100 == 0:
        print(f"{n}/{total} fotogramas")

vid.release()
N_FRAMES = n
ids_brutos = {t for objs in objs_por_frame.values() for t, *_ in objs}
print(f"Pasada 1 terminada: {N_FRAMES} fotogramas, {len(ids_brutos)} IDs brutos del tracker")

100/2832 fotogramas


200/2832 fotogramas


300/2832 fotogramas


400/2832 fotogramas


500/2832 fotogramas


600/2832 fotogramas


... (19 outputs left out)

2600/2832 fotogramas


2700/2832 fotogramas


2800/2832 fotogramas


Pasada 1 terminada: 2832 fotogramas, 247 IDs brutos del tracker


## Unión de pistas (re-identificación a posteriori)

Cada ID bruto es una *pista* (*tracklet*) con un fotograma inicial, uno final, y una primera y última caja. Una
pista **B** se une a una pista anterior **A** cuando:

- son del mismo grupo de clase (persona, moto o vehículo de cuatro ruedas, para tolerar confusiones coche/camión),
- **A** termina antes de que empiece **B**, con un hueco de como mucho `MAX_HUECO` fotogramas,
- la última caja de **A** y la primera de **B** se solapan (IoU ≥ `IOU_MIN`),
- **A** no tiene ya otra sucesora.

Es justo el caso de un coche aparcado que queda tapado por otro que pasa: reaparece en el mismo sitio.
Las cadenas de uniones se resuelven con una estructura de conjuntos disjuntos y cada objeto conserva el ID de su
primera pista.

In [6]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def unir_pistas(objs_por_frame, max_hueco, iou_min):
    """Devuelve {id_bruto: id_unificado}."""
    pistas = {}
    for fr in sorted(objs_por_frame):
        for tid, clase, _, caja in objs_por_frame[fr]:
            p = pistas.setdefault(tid, {"clases": Counter(), "ini": fr, "caja_ini": caja})
            p["clases"][clase] += 1
            p["fin"], p["caja_fin"] = fr, caja
    for p in pistas.values():
        p["grupo"] = GRUPO[p["clases"].most_common(1)[0][0]]

    padre = {t: t for t in pistas}
    con_sucesora = set()
    for b in sorted(pistas, key=lambda t: pistas[t]["ini"]):
        pb = pistas[b]
        mejor = None
        for a, pa in pistas.items():
            if (a == b or a in con_sucesora or pa["grupo"] != pb["grupo"]
                    or not 0 < pb["ini"] - pa["fin"] <= max_hueco):
                continue
            s = iou(pa["caja_fin"], pb["caja_ini"])
            if s >= iou_min and (mejor is None or s > mejor[0]):
                mejor = (s, a)
        if mejor:
            con_sucesora.add(mejor[1])
            padre[b] = mejor[1]

    def raiz(t):
        while padre[t] != t:
            t = padre[t]
        return t
    return {t: raiz(t) for t in pistas}


mapa_id = unir_pistas(objs_por_frame, MAX_HUECO, IOU_MIN)
print(f"IDs brutos: {len(mapa_id)}  ->  objetos tras unir pistas: {len(set(mapa_id.values()))}")

IDs brutos: 247  ->  objetos tras unir pistas: 166


## Objetos válidos y matrículas de cada vehículo

Con los IDs ya unificados se calcula, para cada objeto, su clase (voto mayoritario, para que no parpadee entre coche y
camión), el número de fotogramas en que aparece y el primero, para decidir si cuenta (`MIN_FRAMES`) y para el contador
en pantalla. Después se asigna a cada vehículo su matrícula en cada fotograma; esto no necesita las imágenes.

In [7]:
frames_por_id = Counter()
clases_por_id = defaultdict(Counter)
primer_frame = {}
trayectoria = defaultdict(list)
for fr in sorted(objs_por_frame):
    for tid, clase, _, (x1, y1, x2, y2) in objs_por_frame[fr]:
        u = mapa_id[tid]
        frames_por_id[u] += 1
        clases_por_id[u][clase] += 1
        primer_frame.setdefault(u, fr)
        trayectoria[u].append(((x1 + x2) / 2, (y1 + y2) / 2))

validos = {u for u, c in frames_por_id.items() if c >= MIN_FRAMES}
clase_de = {u: clases_por_id[u].most_common(1)[0][0] for u in frames_por_id}

objetos_por_frame = {}     # fotograma -> [(id, clase, conf, caja)] de objetos válidos
matriculas_por_frame = {}  # fotograma -> {id vehículo: (caja, conf)}
for n in range(1, N_FRAMES + 1):
    objetos = [(mapa_id[t], clase_de[mapa_id[t]], conf, caja) for t, _, conf, caja in objs_por_frame[n]
               if mapa_id[t] in validos]
    objetos_por_frame[n] = objetos
    vehiculos = [(tid, caja) for tid, clase, _, caja in objetos if clase in VEHICULOS]
    matriculas_por_frame[n] = asociar_matriculas(placas_por_frame[n], vehiculos)

n_asociadas = sum(len(m) for m in matriculas_por_frame.values())
con_placa = {tid for m in matriculas_por_frame.values() for tid in m}
print(f"Matrículas asociadas a un vehículo: {n_asociadas} detecciones en {len(con_placa)} vehículos")

Matrículas asociadas a un vehículo: 2652 detecciones en 40 vehículos

## Pasada OCR: lectura de matrículas (P4b)

La comparativa de `P4b/VC_P4b_entrega.ipynb` da dos opciones con ventajas opuestas: **EasyOCR** es rápido (≈70 ms por
recorte) pero falla con placas borrosas; **SmolVLM** acierta mucho más en recortes difíciles (63 % frente a 33 %) pero
tarda ≈1,4 s, así que no se puede pasar por las ≈2600 detecciones de matrícula del vídeo. Se usa un **esquema híbrido**:

1. Cada recorte de matrícula, tomado del fotograma **sin anotar**, se amplía y se lee con EasyOCR (alfabeto `0-9A-Z`).
   Solo se acepta la lectura si tiene formato de matrícula española (`0000BBB` o provincial antiguo). Cada lectura
   suma `conf_ocr × conf_detección` a su texto.
2. Para cada vehículo se guardan sus `VLM_CANDIDATOS` mejores recortes (anchura × nitidez, medida con la varianza del
   Laplaciano). SmolVLM lee solo esos, con el mismo filtro de formato, y cada lectura válida es un voto.
3. El texto del vehículo es el más votado por SmolVLM si lo ha leído al menos `VLM_MIN_VOTOS` veces (o coincide con
   EasyOCR); si no, el de EasyOCR. Una lectura suelta del VLM no basta: con placas borrosas puede inventar caracteres.
4. El texto **solo se muestra desde el primer fotograma en que se leyó**: antes la matrícula aún no era legible.


In [8]:
ALFABETO = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
# Formato actual (0000BBB, sin vocales) y provincial antiguo; igual que en P4b
PATRON_ACTUAL = re.compile(r"\d{4}[BCDFGHJKLMNPRSTVWXYZ]{3}")
PATRON_PROVINCIAL = re.compile(r"[A-Z]{1,3}\d{4}[A-Z]{1,2}")


def normalizar(texto):
    """Mayúsculas, solo alfanuméricos y, si aparece, el fragmento con formato de matrícula."""
    t = re.sub(r"[^0-9A-Z]", "", texto.upper())
    m = PATRON_ACTUAL.search(t) or PATRON_PROVINCIAL.search(t)
    if m:
        return m.group(0)
    return t[1:] if t.startswith("E") and len(t) == 8 else t  # 'E' de la franja azul


def es_matricula(texto):
    return bool(PATRON_ACTUAL.fullmatch(texto) or PATRON_PROVINCIAL.fullmatch(texto))


def recortar(img, caja, margen=MARGEN_RECORTE):
    x1, y1, x2, y2 = caja
    w, h = x2 - x1, y2 - y1
    a, b = max(int(x1 - margen * w), 0), max(int(y1 - margen * h), 0)
    c, d = min(int(x2 + margen * w), img.shape[1]), min(int(y2 + margen * h), img.shape[0])
    return img[b:d, a:c]


def leer_easyocr(reader, recorte):
    """(texto, confianza) si la lectura tiene formato de matrícula; ('', 0) si no."""
    if recorte.size == 0 or recorte.shape[1] < OCR_MIN_ANCHO:
        return "", 0.0
    escala = OCR_ANCHO_OBJETIVO / recorte.shape[1]
    grande = cv2.resize(recorte, None, fx=escala, fy=escala, interpolation=cv2.INTER_CUBIC)
    res = reader.readtext(grande, allowlist=ALFABETO, paragraph=False)
    texto = normalizar("".join(r[1] for r in res))
    if not res or not es_matricula(texto):
        return "", 0.0
    return texto, float(min(r[2] for r in res))


def nitidez(recorte):
    return cv2.Laplacian(cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()

### EasyOCR en cada fotograma

In [9]:
reader = None
if USAR_OCR:
    import easyocr
    reader = easyocr.Reader(["es"], gpu=True, verbose=False)

votos = defaultdict(Counter)    # id -> {texto: suma de conf_ocr * conf_detección}
lectura = {}                    # (fotograma, id) -> texto
mejor_matricula = {}            # id -> (conf, recorte)
candidatos = defaultdict(list)  # id -> montículo de (puntuación, fotograma, recorte)
t_ocr, n_ocr = 0.0, 0

vid = cv2.VideoCapture(str(VIDEO))
for n in range(1, N_FRAMES + 1):
    if not matriculas_por_frame[n]:
        vid.grab()   # avanza sin decodificar: no hay nada que leer
        continue
    ret, img = vid.read()
    if not ret:
        break
    for tid, (caja, mconf) in matriculas_por_frame[n].items():
        rec = recortar(img, caja)
        if tid not in mejor_matricula or mconf > mejor_matricula[tid][0]:
            mejor_matricula[tid] = (mconf, rec.copy())
        if rec.shape[1] >= VLM_MIN_ANCHO:
            heapq.heappush(candidatos[tid], (rec.shape[1] * nitidez(rec) ** 0.5, n, rec.copy()))
            if len(candidatos[tid]) > VLM_CANDIDATOS:
                heapq.heappop(candidatos[tid])
        if reader is None:
            continue
        t0 = time.perf_counter()
        texto, conf_ocr = leer_easyocr(reader, rec)
        t_ocr += time.perf_counter() - t0
        n_ocr += 1
        if texto:
            lectura[(n, tid)] = texto
            votos[tid][texto] += conf_ocr * mconf
vid.release()
reader = None

texto_easy = {tid: c.most_common(1)[0][0] for tid, c in votos.items() if c}
if n_ocr:
    print(f"EasyOCR: {n_ocr} recortes en {t_ocr:.0f} s ({1000 * t_ocr / n_ocr:.0f} ms/recorte), "
          f"{len(lectura)} lecturas con formato válido, {len(texto_easy)} vehículos leídos")

EasyOCR: 2652 recortes en 33 s (12 ms/recorte), 12 lecturas con formato válido, 2 vehículos leídos


### SmolVLM sobre los mejores recortes y texto final

In [10]:
votos_vlm = defaultdict(Counter)
lectura_vlm = {}
t_vlm, n_vlm = 0.0, 0
if USAR_VLM and candidatos:
    import torch
    from PIL import Image
    from transformers import AutoModelForImageTextToText, AutoProcessor

    torch.cuda.empty_cache()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(VLM_MODELO)
    vlm = AutoModelForImageTextToText.from_pretrained(
        VLM_MODELO, dtype=torch.bfloat16 if dev == "cuda" else torch.float32,
        _attn_implementation="eager").to(dev).eval()
    prompt = processor.apply_chat_template([{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Read the license plate in the image. Answer only with its characters."}]}],
        add_generation_prompt=True)

    def leer_vlm(img):
        pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        entrada = processor(text=prompt, images=[pil], return_tensors="pt").to(dev)
        with torch.no_grad():
            ids = vlm.generate(**entrada, max_new_tokens=16, do_sample=False)
        return processor.batch_decode(ids[:, entrada["input_ids"].shape[1]:], skip_special_tokens=True)[0]

    for tid, cands in candidatos.items():
        for _, fr, rec in sorted(cands, key=lambda c: c[1]):
            t0 = time.perf_counter()
            texto = normalizar(leer_vlm(rec))
            t_vlm += time.perf_counter() - t0
            n_vlm += 1
            if es_matricula(texto):
                votos_vlm[tid][texto] += 1
                lectura_vlm[(fr, tid)] = texto
    del vlm
    torch.cuda.empty_cache()
    print(f"SmolVLM: {n_vlm} recortes en {t_vlm:.0f} s ({t_vlm / max(n_vlm, 1):.2f} s/recorte), "
          f"{len(lectura_vlm)} lecturas con formato válido")

texto_vlm = {}
for tid, c in votos_vlm.items():
    texto, n_votos = c.most_common(1)[0]
    if n_votos >= VLM_MIN_VOTOS or texto == texto_easy.get(tid):
        texto_vlm[tid] = texto
texto_de = {**texto_easy, **texto_vlm}
fuente_de = {tid: "smolvlm" if tid in texto_vlm else "easyocr" for tid in texto_de}
desde = {}
for (fr, tid), t in list(lectura.items()) + list(lectura_vlm.items()):
    if texto_de.get(tid) == t:
        desde[tid] = min(fr, desde.get(tid, fr))

print(f"Vehículos con matrícula leída: {len(texto_de)} de {len(con_placa)} con matrícula detectada")
for tid, texto in sorted(texto_de.items()):
    print(f"  #{tid} ({clase_de[tid]}): {texto:8} [{fuente_de[tid]}] desde el fotograma {desde[tid]} | "
          f"EasyOCR {dict(votos[tid].most_common(2))} | SmolVLM {dict(votos_vlm[tid])}")

E:\conda\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SmolVLM: 203 recortes en 302 s (1.49 s/recorte), 66 lecturas con formato válido
Vehículos con matrícula leída: 11 de 40 con matrícula detectada
  #60 (car): 3713LKC  [smolvlm] desde el fotograma 783 | EasyOCR {} | SmolVLM {'3713LKC': 7}
  #98 (car): 6713HCR  [smolvlm] desde el fotograma 1291 | EasyOCR {} | SmolVLM {'6713HCR': 4, '6713HKP': 1}
  #134 (car): 2914FGN  [smolvlm] desde el fotograma 1356 | EasyOCR {} | SmolVLM {'2314FGN': 1, '2914FGN': 2, '2916FCN': 1, 'Z2164FE': 1}
  #151 (car): 0053JFW  [smolvlm] desde el fotograma 1459 | EasyOCR {} | SmolVLM {'0053JFW': 5}
  #161 (car): 8098JXM  [smolvlm] desde el fotograma 2349 | EasyOCR {'2098JXV': 0.8694531294758252} | SmolVLM {'8098JXM': 2}
  #165 (car): 3902FXD  [smolvlm] desde el fotograma 1639 | EasyOCR {} | SmolVLM {'3902FXD': 2, '3802FXD': 1, '3902FWD': 1, '3902FMD': 1}
  #190 (car): 8536LKJ  [smolvlm] desde el fotograma 1786 | EasyOCR {} | SmolVLM {'8536LKJ': 2}
  #204 (car): 9059LRY  [smolvlm] desde el fotograma 1706 | EasyOCR 

## Pasada 2: vídeo, CSV y vídeo anonimizado

Se vuelve a leer el vídeo y se dibuja y escribe el CSV con los IDs unificados. El texto de la matrícula que se muestra
y se guarda (`texto_matricula`) es el final del vehículo, **a partir del primer fotograma en que se leyó**: así
se mantiene aunque la placa deje de verse, pero no aparece antes de poder leerse. `lectura_fotograma` guarda lo
leído solo en ese fotograma (por EasyOCR o SmolVLM). El vídeo anonimizado se genera a partir del fotograma
original, antes de dibujar nada, difuminando personas y todas las matrículas detectadas.

In [11]:
CAMPOS = ["fotograma", "tipo_objeto", "confianza", "identificador_tracking", "x1", "y1", "x2", "y2",
          "matricula_en_su_caso", "confianza_matricula", "mx1", "my1", "mx2", "my2", "texto_matricula",
          "lectura_fotograma"]

COLORES = {"person": (0, 200, 0), "car": (255, 128, 0), "motorcycle": (0, 200, 255),
           "bus": (200, 0, 200), "truck": (0, 0, 255)}
COLOR_PLACA = (255, 0, 255)


def difuminar(img, caja):
    x1, y1, x2, y2 = (max(0, int(v)) for v in caja)
    roi = img[y1:y2, x1:x2]
    if roi.size:
        k = max(15, (min(roi.shape[:2]) // 3) | 1)
        img[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (k, k), 0)


vid = cv2.VideoCapture(str(VIDEO))
fps = vid.get(cv2.CAP_PROP_FPS) or 25
ancho = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(VIDEO_OUT), fourcc, fps, (ancho, alto))
out_anon = cv2.VideoWriter(str(VIDEO_ANON), fourcc, fps, (ancho, alto)) if ANONIMIZAR else None

with open(CSV_OUT, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=CAMPOS)
    escritor.writeheader()
    for n in range(1, N_FRAMES + 1):
        ret, img = vid.read()
        if not ret:
            break
        objetos = objetos_por_frame[n]
        matriculas = matriculas_por_frame[n]

        if out_anon is not None:   # antes de dibujar encima
            anon = img.copy()
            for _, clase, _, caja in objetos:
                if clase == "person":
                    difuminar(anon, caja)
            for caja, _ in placas_por_frame[n]:
                difuminar(anon, caja)
            out_anon.write(anon)

        for tid, clase, conf, (x1, y1, x2, y2) in objetos:
            texto = texto_de.get(tid, "") if n >= desde.get(tid, N_FRAMES + 1) else ""
            fila = {"fotograma": n, "tipo_objeto": clase, "confianza": round(conf, 3), "identificador_tracking": tid,
                    "x1": round(x1), "y1": round(y1), "x2": round(x2), "y2": round(y2),
                    "matricula_en_su_caso": 0, "confianza_matricula": "", "mx1": "", "my1": "",
                    "mx2": "", "my2": "", "texto_matricula": texto, "lectura_fotograma": ""}

            color = COLORES[clase]
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(img, f"{clase} #{tid} {conf:.2f}", (int(x1), max(int(y1) - 6, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            if tid in matriculas:
                (mx1, my1, mx2, my2), mconf = matriculas[tid]
                fila.update({"matricula_en_su_caso": 1, "confianza_matricula": round(mconf, 3),
                             "mx1": round(mx1), "my1": round(my1), "mx2": round(mx2), "my2": round(my2),
                             "lectura_fotograma": lectura_vlm.get((n, tid)) or lectura.get((n, tid), "")})
                cv2.rectangle(img, (int(mx1), int(my1)), (int(mx2), int(my2)), COLOR_PLACA, 2)

            if texto:
                (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
                cv2.rectangle(img, (int(x1), int(y2)), (int(x1) + tw + 8, int(y2) + th + 10), (0, 0, 0), -1)
                cv2.putText(img, texto, (int(x1) + 4, int(y2) + th + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                            (0, 255, 255), 2)

            escritor.writerow(fila)

        hasta_ahora = Counter(clase_de[u] for u in validos if primer_frame[u] <= n)
        texto_conteo = "  ".join(f"{c}: {hasta_ahora.get(c, 0)}" for c in CLASES.values())
        cv2.rectangle(img, (0, 0), (ancho, 34), (0, 0, 0), -1)
        cv2.putText(img, texto_conteo, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(img)
        if SHOW:
            cv2.imshow("Seguimiento", img)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

vid.release()
out.release()
if out_anon is not None:
    out_anon.release()
cv2.destroyAllWindows()
print("Vídeo:", VIDEO_OUT.resolve())
if ANONIMIZAR:
    print("Vídeo anonimizado:", VIDEO_ANON.resolve())
print("CSV:  ", CSV_OUT.resolve())

Vídeo: D:\CarpetaDescargasC\Documentos\AaCurso27\VC\Practica4\VC_P4\P4\resultado_seguimiento\seguimiento.mp4
Vídeo anonimizado: D:\CarpetaDescargasC\Documentos\AaCurso27\VC\Practica4\VC_P4\P4\resultado_seguimiento\seguimiento_anonimo.mp4
CSV:   D:\CarpetaDescargasC\Documentos\AaCurso27\VC\Practica4\VC_P4\P4\resultado_seguimiento\detecciones.csv


## Conteo por clase

Se cuentan **objetos únicos tras la unión de pistas**, no detecciones ni IDs brutos. Cada objeto se asigna a la clase
que más veces se le ha detectado y solo se cuenta si aparece en al menos `MIN_FRAMES` fotogramas. Aun así el
conteo es una estimación: dos objetos distintos que se cruzan pueden fusionarse, y una oclusión muy larga puede
dejar un objeto partido en dos. Conviene comprobar el vídeo a mano y comentar estos errores en el README.

In [12]:
conteo = Counter(clase_de[u] for u in validos)

print(f"IDs brutos del tracker: {len(ids_brutos)}")
print(f"Objetos tras unir pistas: {len(frames_por_id)}  |  con >= {MIN_FRAMES} fotogramas: {len(validos)}")
for clase in CLASES.values():
    print(f"{clase:>12}: {conteo.get(clase, 0)}")

with open(CONTEO_OUT, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["clase", "total_objetos_unicos"])
    for clase in CLASES.values():
        w.writerow([clase, conteo.get(clase, 0)])

with open(MATRICULAS_OUT, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["identificador_tracking", "tipo_objeto", "texto_matricula", "fuente", "primer_fotograma_leido",
                "texto_easyocr", "lecturas_easyocr", "texto_smolvlm", "lecturas_smolvlm",
                "fotogramas_con_matricula", "mejor_confianza_deteccion"])
    fotos_placa = Counter(tid for m in matriculas_por_frame.values() for tid in m)
    lecturas_id = Counter(tid for _, tid in lectura)
    for tid in sorted(con_placa):
        w.writerow([tid, clase_de[tid], texto_de.get(tid, ""), fuente_de.get(tid, ""), desde.get(tid, ""),
                    texto_easy.get(tid, ""), lecturas_id[tid], texto_vlm.get(tid, ""), sum(votos_vlm[tid].values()),
                    fotos_placa[tid], round(mejor_matricula[tid][0], 3)])
print(f"Matrículas leídas: {len(texto_de)} -> {MATRICULAS_OUT.name}")

for viejo in RECORTES.glob("*.jpg"):   # recortes de ejecuciones anteriores
    viejo.unlink()
guardados = 0
for tid, (mconf, recorte) in mejor_matricula.items():
    if recorte.size:
        sufijo = f"_{texto_de[tid]}" if tid in texto_de else ""
        cv2.imwrite(str(RECORTES / f"id{tid:04d}_{mconf:.2f}{sufijo}.jpg"), recorte)
        guardados += 1
print(f"Recortes de matrícula guardados: {guardados} en {RECORTES.resolve()}")

IDs brutos del tracker: 247
Objetos tras unir pistas: 166  |  con >= 10 fotogramas: 149
      person: 51
         car: 93
  motorcycle: 2
         bus: 2
       truck: 1
Matrículas leídas: 11 -> matriculas_leidas.csv
Recortes de matrícula guardados: 40 en D:\CarpetaDescargasC\Documentos\AaCurso27\VC\Practica4\VC_P4\P4\resultado_seguimiento\matriculas


## Extra: flujo direccional

Se compara el centro de cada objeto al principio y al final de su recorrido. Solo se consideran objetos válidos
con un desplazamiento mínimo (`UMBRAL`, en píxeles) para ignorar los que están prácticamente quietos.

In [13]:
UMBRAL = 80

def direccion(puntos):
    (x0, y0), (x1, y1) = puntos[0], puntos[-1]
    dx, dy = x1 - x0, y1 - y0
    if max(abs(dx), abs(dy)) < UMBRAL:
        return "estático"
    if abs(dx) >= abs(dy):
        return "derecha" if dx > 0 else "izquierda"
    return "abajo (se acerca)" if dy > 0 else "arriba (se aleja)"

flujo = defaultdict(Counter)
for u in validos:
    flujo[clase_de[u]][direccion(trayectoria[u])] += 1

for clase, c in flujo.items():
    print(clase, dict(c))

with open(FLUJO_OUT, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["identificador_tracking", "tipo_objeto", "fotogramas", "direccion", "texto_matricula"])
    for u in sorted(validos):
        w.writerow([u, clase_de[u], frames_por_id[u], direccion(trayectoria[u]), texto_de.get(u, "")])
print("Flujo por objeto:", FLUJO_OUT.resolve())

bus {'izquierda': 1, 'estático': 1}
car {'izquierda': 37, 'estático': 47, 'derecha': 9}
person {'derecha': 19, 'estático': 25, 'izquierda': 7}
motorcycle {'izquierda': 2}
truck {'estático': 1}
Flujo por objeto: D:\CarpetaDescargasC\Documentos\AaCurso27\VC\Practica4\VC_P4\P4\resultado_seguimiento\flujo.csv
